# IOAI — 2025 Summer National Classifier Clone (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad haio
!cd haio && git sparse-checkout set 2025/nyari-orszagos/feladatok/adatok/klasszifikalo-klon >/dev/null && git checkout -q
import shutil, glob
for f in glob.glob('haio/2025/nyari-orszagos/feladatok/adatok/klasszifikalo-klon/*.pt'): shutil.copy(f, '.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 분류기 복제 (Classifier Clone) — 모범답안

HAIO 2025 여름 결선 (NN). 블랙박스 **신경망**(`secret_model.pt`, 11→6 클래스)을 **비신경망 모델**로 복제한다.
점수 = **일치율(agreement)** — 내 모델의 예측이 신경망의 예측과 얼마나 같은가(테스트 320개).
제출 `submission.csv`(id, label).

**핵심 기법 — 질의 증강(query augmentation)**: 이것은 **모델 추출(model extraction)** 이다. 신경망은 함수이므로,
그 함수를 **많은 입력점에서 질의**해 (입력, 신경망예측) 쌍을 대량으로 만들면 비신경망 모델이 그 결정경계를
촘촘히 배울 수 있다. 학습 데이터 주변에 **가우시안 섭동 + 볼록 보간 + 특징박스 샘플** 을 8만여 개 생성해 신경망에
질의 → **HistGradientBoosting** 으로 증류(distill). → 일치율 ≈ **0.83** (원 학습셋만 쓰는 RF 베이스라인 ≈0.69).

*(비신경망 제약 준수: HistGradientBoosting=트리 부스팅. 신경망 자체 예측을 그대로 베끼는 건 복제가 아니므로
학습 영역에서만 질의해 일반화하는 정직한 증류를 한다.)*


In [ ]:
import numpy as np, torch, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

secret = torch.jit.load("secret_model.pt", map_location="cpu"); secret.eval()   # 복제 대상(블랙박스 NN)
sp = torch.load("train_test_split.pt", map_location="cpu", weights_only=False)
X_train, X_test = sp[0].float().numpy(), sp[1].float().numpy()

@torch.no_grad()
def nn_predict(X):                                   # 신경망 argmax (teacher)
    return secret(torch.as_tensor(X, dtype=torch.float32)).argmax(1).numpy()
print("train", X_train.shape, "test", X_test.shape, "| classes", secret(torch.as_tensor(X_train[:2])).shape[1])


In [ ]:
# 질의 증강: 학습 영역 주변에 합성점 생성 → 신경망에 질의해 라벨 확보
rng = np.random.RandomState(0)
std = X_train.std(0); lo, hi = X_train.min(0), X_train.max(0)
parts = [X_train]
for s in np.linspace(0.02, 0.4, 20):                 # 다중스케일 가우시안 섭동(데이터 매니폴드 주변)
    parts.append(X_train + rng.randn(*X_train.shape) * std * s)
n = 40000                                            # 볼록 보간(두 학습점 사이)
a = X_train[rng.randint(0, len(X_train), n)]; b = X_train[rng.randint(0, len(X_train), n)]
t = rng.rand(n, 1); parts.append(a * t + b * (1 - t))
parts.append(lo + rng.rand(15000, X_train.shape[1]) * (hi - lo))   # 특징 박스 커버리지
X_aug = np.vstack(parts).astype(np.float32)
y_aug = nn_predict(X_aug)                             # 신경망에 질의 → 라벨
print("증강 질의셋:", X_aug.shape)


In [ ]:
# 비신경망 모델(HistGradientBoosting)로 증류 → 테스트 예측
clf = HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.1, max_leaf_nodes=255, random_state=0)
clf.fit(X_aug, y_aug)
pred = clf.predict(X_test)
pd.DataFrame({"id": range(len(pred)), "label": pred}).to_csv("submission.csv", index=False)
# 참고: 학습셋에 대한 신경망과의 일치율(자기점검)
print("submission.csv 저장:", len(pred), "| train agreement", round(accuracy_score(nn_predict(X_train), clf.predict(X_train)), 4))


### 정리
- **질의 증강 + HistGradientBoosting 증류** → 테스트 일치율 ≈ **0.83** (원 학습셋만 RF ≈0.69).
- **핵심**: 신경망=함수이므로 입력공간을 촘촘히 질의(가우시안 섭동·보간·박스)해 (입력,예측) 쌍을 대량 확보하면
  트리 부스팅이 그 결정경계를 잘 근사한다(모델 추출). 섭동 폭이 너무 크면 매니폴드를 벗어나 오히려 나빠진다.
- **더 시도해볼 것**: 소프트라벨(로짓) 회귀 증류·경계 근처 능동 질의(active learning)·앙상블.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)